## 0. Google Colab Setup

Mount Google Drive to access the project directory. Run this cell first in every session to establish the working path.

In [1]:
from google.colab import drive
# Mount to the standard base directory
drive.mount('/content/drive')

# Now you can define your project path and use it
project_path = '/content/drive/MyDrive/multimodal-causal-ablation'
import os
if os.path.exists(project_path):
    print(f'Successfully accessed: {project_path}')
else:
    print(f'Drive mounted, but folder not found: {project_path}')

Mounted at /content/drive
Successfully accessed: /content/drive/MyDrive/multimodal-causal-ablation


# Phase A — Dominant Modality Verification

Verify Audio as the dominant modality via aggregated DeepSHAP attribution, following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Formally confirm which modality contributes the highest aggregated DeepSHAP attribution score for both the base and fine-tuned models. This is the prerequisite gate before any neuron-level probing or ablation work in Phases B–D.

## 1. Environment & Imports

Set up the deterministic seed (`seed=0`, matching upstream checkpoint convention) and import all required libraries. The seed utility from `src/utils.py` pins `torch`, `numpy`, `random`, and `cudnn` for full reproducibility across ephemeral Colab runtimes.

**Expected output:** Confirmation of project path, checkpoints directory, and results directory.

In [2]:
import sys
import os
import pickle

import numpy as np
import pandas as pd

# Add project src to path for utility imports
sys.path.insert(0, os.path.join(project_path, 'src'))
from utils import set_deterministic_seed

# Pin all randomness sources (seed=0 matches upstream checkpoint convention)
set_deterministic_seed(seed=0)

# Define paths
checkpoints_dir = os.path.join(project_path, 'checkpoints')
results_dir = os.path.join(project_path, 'results')
os.makedirs(results_dir, exist_ok=True)

print(f'Project path:  {project_path}')
print(f'Checkpoints:   {checkpoints_dir}')
print(f'Results:       {results_dir}')

Project path:  /content/drive/MyDrive/multimodal-causal-ablation
Checkpoints:   /content/drive/MyDrive/multimodal-causal-ablation/checkpoints
Results:       /content/drive/MyDrive/multimodal-causal-ablation/results


## 2. Load Pre-computed DeepSHAP Attributions

Load the pre-computed DeepSHAP attribution pickles for both the base and fine-tuned models. Each pickle contains a dict with:
- `'SHAP_value'`: list of 6 numpy arrays (one per emotion class), each shaped `(n_samples, 1152)`
- `'test_feature'`: corresponding test input features

The 1152-dimensional feature vector is a concatenation of three modality representations:
- **Text** (ALBERT): dimensions 0–1023 (1024-d)
- **Video** (visual): dimensions 1024–1087 (64-d)
- **Audio** (acoustic): dimensions 1088–1151 (64-d)

**Expected output:** Structure summary showing keys, number of classes, and per-class array shapes for both models.

In [3]:
# Load pre-computed DeepSHAP attributions for both models
with open(os.path.join(checkpoints_dir, 'base_shap.pkl'), 'rb') as f:
    base_shap_data = pickle.load(f)

with open(os.path.join(checkpoints_dir, 'finetuned_shap.pkl'), 'rb') as f:
    finetuned_shap_data = pickle.load(f)

# Inspect structure
print('=== Base Model SHAP ===')
print(f'Keys: {list(base_shap_data.keys())}')
print(f'Number of classes: {len(base_shap_data["SHAP_value"])}')
for i, sv in enumerate(base_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

print()
print('=== Fine-tuned Model SHAP ===')
print(f'Keys: {list(finetuned_shap_data.keys())}')
print(f'Number of classes: {len(finetuned_shap_data["SHAP_value"])}')
for i, sv in enumerate(finetuned_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

=== Base Model SHAP ===
Keys: ['SHAP_value', 'test_feature']
Number of classes: 6
  Class 0: shape = (144, 1152)
  Class 1: shape = (144, 1152)
  Class 2: shape = (144, 1152)
  Class 3: shape = (144, 1152)
  Class 4: shape = (144, 1152)
  Class 5: shape = (144, 1152)

=== Fine-tuned Model SHAP ===
Keys: ['SHAP_value', 'test_feature']
Number of classes: 6
  Class 0: shape = (144, 1152)
  Class 1: shape = (144, 1152)
  Class 2: shape = (144, 1152)
  Class 3: shape = (144, 1152)
  Class 4: shape = (144, 1152)
  Class 5: shape = (144, 1152)


## 3. Compute Aggregated SHAP Attribution per Modality

Raw 1152-d SHAP values create a **dimensionality illusion**: Text has 16× more features than Audio or Video, so naive per-feature comparisons inflate Text's apparent contribution.

Per ADR 0001, I resolve this by computing **aggregated** SHAP attribution:
1. For each emotion class, take the absolute value of all SHAP values.
2. Sum |SHAP| within each modality for every sample — this collapses each modality's contribution to a single scalar per sample.
3. Average across samples to get one attribution score per modality per class.
4. Average across classes to get the overall modality attribution.

This makes the comparison fair regardless of how many raw features each modality contributes.

**Expected output:** Per-class attribution percentages for Text, Video, and Audio in both models.

In [ ]:
# Modality feature ranges in the 1152-d concatenated feature vector
MODALITY_RANGES = {
    'Text':  (0, 1024),     # ALBERT embeddings (1024-d)
    'Video': (1024, 1088),  # Visual features (64-d)
    'Audio': (1088, 1152),  # Acoustic features (64-d)
}

EMOTION_CLASSES = [
    'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise'
]


def compute_aggregated_shap(shap_data, model_name):
    """Compute aggregated SHAP attribution per modality.

    For each class, sum |SHAP values| within each modality per sample,
    then average across samples. This is the aggregation method from
    ADR 0001 to resolve the dimensionality illusion between Text
    (1024-d) and Audio (64-d).
    """
    shap_values = shap_data['SHAP_value']
    n_classes = len(shap_values)

    rows = []
    for class_idx in range(n_classes):
        sv = shap_values[class_idx]  # (n_samples, 1152)

        row = {'model': model_name, 'class': EMOTION_CLASSES[class_idx]}
        for mod_name, (start, end) in MODALITY_RANGES.items():
            # Sum |SHAP| within modality per sample, then mean across samples
            per_sample = np.sum(np.abs(sv[:, start:end]), axis=1)
            row[f'{mod_name}_attribution'] = np.mean(per_sample)
            row[f'{mod_name}_std'] = np.std(per_sample)

        # Percentages for readability
        total = sum(row[f'{m}_attribution'] for m in MODALITY_RANGES)
        for mod_name in MODALITY_RANGES:
            row[f'{mod_name}_pct'] = (
                row[f'{mod_name}_attribution'] / total * 100
            )

        rows.append(row)

    return pd.DataFrame(rows)


# Compute for both models
base_attr_df = compute_aggregated_shap(base_shap_data, 'base')
finetuned_attr_df = compute_aggregated_shap(finetuned_shap_data, 'finetuned')

# Combine results
attribution_df = pd.concat(
    [base_attr_df, finetuned_attr_df], ignore_index=True
)

# Display per-class results
print('=== Per-Class Aggregated SHAP Attribution (%) ===')
print()
display_cols = ['model', 'class', 'Text_pct', 'Video_pct', 'Audio_pct']
print(
    attribution_df[display_cols].to_string(
        index=False, float_format='%.2f'
    )
)

=== Per-Class Aggregated SHAP Attribution (%) ===

    model     class  Text_pct  Video_pct  Audio_pct
     base     anger     71.20       2.65      26.15
     base   disgust     74.49       3.50      22.01
     base      fear     75.74       2.89      21.37
     base happiness     70.90       3.35      25.76
     base   sadness     72.32       2.82      24.85
     base  surprise     73.73       2.60      23.67
finetuned     anger     66.99       1.39      31.62
finetuned   disgust     68.20       2.00      29.80
finetuned      fear     70.55       1.27      28.19
finetuned happiness     68.54       2.04      29.41
finetuned   sadness     67.43       1.67      30.90
finetuned  surprise     70.08       1.10      28.82


## 4. Formal Dominant Modality Verdict

Apply the decision rule from ADR 0001:
1. Rank modalities by their overall aggregated SHAP attribution (averaged across all 6 emotion classes).
2. If the gap between the top two modalities is ≤ 5%, treat them as within parity and target **Audio** (64-d) for its superior neuron-to-class ratio.
3. Otherwise, the highest-ranked modality is the Dominant Modality.

**Outputs saved:**
- `results/phase_a_dominant_modality_verdict.csv` — Overall verdict per model
- `results/phase_a_shap_attribution_by_class.csv` — Detailed per-class breakdown

**Expected output:** Ranked modality percentages and formal verdict for both models.

In [5]:
def compute_verdict(attr_df, model_name):
    """Apply ADR 0001 dominant modality decision rule.

    Returns a dict with the verdict and supporting evidence.
    """
    model_df = attr_df[attr_df['model'] == model_name]

    # Overall attribution: mean across classes
    summary = {}
    for mod_name in MODALITY_RANGES:
        summary[mod_name] = model_df[f'{mod_name}_attribution'].mean()

    total = sum(summary.values())
    pct = {k: v / total * 100 for k, v in summary.items()}

    # Rank by attribution
    ranked = sorted(pct.items(), key=lambda x: x[1], reverse=True)

    sep = '=' * 55
    print(sep)
    print(f'  {model_name.upper()} MODEL — Aggregated SHAP Attribution')
    print(sep)
    for mod, p in ranked:
        print(f'  {mod:8s}: {p:6.2f}%  (raw mean: {summary[mod]:.6f})')

    top_mod, top_pct = ranked[0]
    second_mod, second_pct = ranked[1]
    gap = top_pct - second_pct

    # ADR 0001 decision rule
    if gap <= 5.0:
        dominant = 'Audio'
        reason = (
            f'Top two modalities ({top_mod}: {top_pct:.2f}%, '
            f'{second_mod}: {second_pct:.2f}%) are within 5% parity '
            f'(gap = {gap:.2f}%). Per ADR 0001, targeting Audio (64-d) '
            f'for superior neuron-to-class ratio.'
        )
    else:
        dominant = top_mod
        reason = (
            f'{top_mod} leads with {top_pct:.2f}% vs '
            f'{second_mod} at {second_pct:.2f}% '
            f'(gap = {gap:.2f}% > 5% threshold).'
        )

    print()
    print(f'  VERDICT: Dominant Modality = {dominant}')
    print(f'  Reason:  {reason}')

    return {
        'model': model_name,
        'dominant_modality': dominant,
        'Text_pct': round(pct['Text'], 4),
        'Video_pct': round(pct['Video'], 4),
        'Audio_pct': round(pct['Audio'], 4),
        'top_modality': top_mod,
        'second_modality': second_mod,
        'gap_pct': round(gap, 4),
        'parity_rule_applied': gap <= 5.0,
        'reason': reason,
    }


# Apply verdict to both models
base_verdict = compute_verdict(attribution_df, 'base')
print()
finetuned_verdict = compute_verdict(attribution_df, 'finetuned')

# --- Save results ---
verdict_df = pd.DataFrame([base_verdict, finetuned_verdict])
verdict_path = os.path.join(
    results_dir, 'phase_a_dominant_modality_verdict.csv'
)
verdict_df.to_csv(verdict_path, index=False)

detail_path = os.path.join(
    results_dir, 'phase_a_shap_attribution_by_class.csv'
)
attribution_df.to_csv(detail_path, index=False)

print()
print(f'Results saved:')
print(f'  Verdict:  {verdict_path}')
print(f'  Details:  {detail_path}')

# --- Summary ---
sep = '=' * 55
print()
print(sep)
print('  PHASE A SUMMARY')
print(sep)
bm = base_verdict['dominant_modality']
fm = finetuned_verdict['dominant_modality']
print(f'  Base model dominant modality:       {bm}')
print(f'  Fine-tuned model dominant modality:  {fm}')

if base_verdict['dominant_modality'] == 'Audio':
    print()
    print('  ✓ Audio confirmed as dominant modality.')
    print('    Proceed to Phase B: Probe Signal Validation '
          'on Audio FFN activations.')
else:
    dm = base_verdict['dominant_modality']
    print()
    print(f'  ✗ Audio NOT confirmed. Dominant modality = {dm}.')
    print('    Review methodology — experiment targets '
          'the dominant modality.')

  BASE MODEL — Aggregated SHAP Attribution
  Text    :  73.17%  (raw mean: 0.258486)
  Audio   :  23.84%  (raw mean: 0.084228)
  Video   :   2.99%  (raw mean: 0.010561)

  VERDICT: Dominant Modality = Text
  Reason:  Text leads with 73.17% vs Audio at 23.84% (gap = 49.33% > 5% threshold).

  FINETUNED MODEL — Aggregated SHAP Attribution
  Text    :  68.67%  (raw mean: 0.265253)
  Audio   :  29.77%  (raw mean: 0.114995)
  Video   :   1.56%  (raw mean: 0.006042)

  VERDICT: Dominant Modality = Text
  Reason:  Text leads with 68.67% vs Audio at 29.77% (gap = 38.90% > 5% threshold).

Results saved:
  Verdict:  /content/drive/MyDrive/multimodal-causal-ablation/results/phase_a_dominant_modality_verdict.csv
  Details:  /content/drive/MyDrive/multimodal-causal-ablation/results/phase_a_shap_attribution_by_class.csv

  PHASE A SUMMARY
  Base model dominant modality:       Text
  Fine-tuned model dominant modality:  Text

  ✗ Audio NOT confirmed. Dominant modality = Text.
    Review methodology —